In [ ]:
import h5py
import numpy as np
from pathlib import Path
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import gaussian_filter
from tqdm.auto import tqdm

# --- Configuration ---
DATA_DIR = Path.cwd().parent.parent / "data" / "darcy_flow_data"
IN_FILE_PATH = DATA_DIR / "2D_DarcyFlow_beta10.0_Train.hdf5"

scale_factor = 1
OUT_FILE_PATH = DATA_DIR / f"2D_DarcyFlow_beta10.0_Train_Scale{scale_factor}.hdf5"

print(f"Generating scaled dataset (Factor: {scale_factor}) -> {OUT_FILE_PATH.name}")

with h5py.File(IN_FILE_PATH, 'r') as f_in:
    x_coords = np.array(f_in['x-coordinate'][:])
    y_coords = np.array(f_in['y-coordinate'][:])
    N_samples = f_in['nu'].shape[0]
    
    # Grid setup
    nx, ny = len(x_coords), len(y_coords)
    fine_nx, fine_ny = nx * scale_factor, ny * scale_factor
    
    x_l, x_u = float(np.min(x_coords)), float(np.max(x_coords)) 
    y_l, y_u = float(np.min(y_coords)), float(np.max(y_coords)) 
    
    fine_x_coords = np.linspace(x_l, x_u, fine_nx)
    fine_y_coords = np.linspace(y_l, y_u, fine_ny)
    X_fine, Y_fine = np.meshgrid(fine_x_coords, fine_y_coords, indexing='ij')
    
    h_x_fine = fine_x_coords[1] - fine_x_coords[0]
    h_y_fine = fine_y_coords[1] - fine_y_coords[0]
    
    # --- Stream and Write ---
    with h5py.File(OUT_FILE_PATH, 'w') as f_out:
        # Save the new coordinate grid
        f_out.create_dataset('x-coordinate', data=fine_x_coords)
        f_out.create_dataset('y-coordinate', data=fine_y_coords)
        
        # Create empty datasets ready to accept flattened (N_points, 1) arrays
        # Use float64 to maintain precision for the direct solver
        shape_flat = (N_samples, fine_nx * fine_ny, 1)
        ds_a = f_out.create_dataset('a_flat', shape=shape_flat, dtype=np.float64)
        ds_ax = f_out.create_dataset('a_x_flat', shape=shape_flat, dtype=np.float64)
        ds_ay = f_out.create_dataset('a_y_flat', shape=shape_flat, dtype=np.float64)
        ds_u = f_out.create_dataset('u_flat', shape=shape_flat, dtype=np.float64)
        
        # Process sample-by-sample to keep RAM near 0
        for i in tqdm(range(N_samples), desc="Processing Samples"):
            a_raw = f_in['nu'][i]
            u_raw = f_in['tensor'][i].squeeze()
            
            # 1. Interpolate
            interp_a = RegularGridInterpolator((x_coords, y_coords), a_raw, method='nearest')
            a_fine = interp_a((X_fine, Y_fine))
            
            interp_u = RegularGridInterpolator((x_coords, y_coords), u_raw, method='linear')
            u_fine = interp_u((X_fine, Y_fine))
            
            # 2. Smooth 'a'
            a_smoothed = gaussian_filter(a_fine, sigma=scale_factor)
            
            # 3. Calculate spatial gradients
            a_x, a_y = np.gradient(a_smoothed, h_x_fine, h_y_fine)
            
            # 4. Flatten and write directly to disk
            ds_a[i] = a_smoothed.reshape(-1, 1)
            ds_ax[i] = a_x.reshape(-1, 1)
            ds_ay[i] = a_y.reshape(-1, 1)
            ds_u[i] = u_fine.reshape(-1, 1)

print("Dataset generation complete!")